## data ingestion & cleaning

In [35]:
from datasets import load_dataset, concatenate_datasets

dataset = load_dataset("theatticusproject/cuad-qa", revision="refs/convert/parquet")
train_data = dataset["train"]
test_data = dataset["test"]
print(len(train_data))
print(len(test_data))

22450
4182


In [36]:
full_data = concatenate_datasets([train_data, test_data])

In [37]:
print(len(full_data))

26632


In [38]:
full_data

Dataset({
    features: ['id', 'title', 'context', 'question', 'answers'],
    num_rows: 26632
})

In [39]:
seen_titles = set()
unique_contracts = []

for row in full_data:
    if row["title"] not in seen_titles:
        seen_titles.add(row["title"])
        unique_contracts.append({
            "title": row["title"],
            "context": row["context"]
        })
print(f"Number of unique contracts: {len(unique_contracts)}")

Number of unique contracts: 510


In [40]:
import pandas as pd
df = pd.DataFrame(unique_contracts)
print(df["context"].str.len().describe())

count       510.000000
mean      52563.005882
std       55946.484865
min         645.000000
25%       16416.250000
50%       33143.000000
75%       66394.750000
max      338211.000000
Name: context, dtype: float64


In [41]:
print(df["context"].head())

0    EXHIBIT 10.6\n\n                              ...
1    Exhibit 10.26    CONFIDENTIAL TREATMENT HAS BE...
2    Exhibit 1\n\nJOINT FILING AGREEMENT\n\nThe und...
3    REDACTED COPY\n\nCONFIDENTIAL TREATMENT REQUES...
4    Exhibit 10.23 Corporate Address Fannin South P...
Name: context, dtype: object


In [42]:
df.head()

,title,context
0,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,EXHIBIT 10.6\n\n ...
1,"WHITESMOKE,INC_11_08_2011-EX-10.26-PROMOTION A...",Exhibit 10.26 CONFIDENTIAL TREATMENT HAS BE...
2,NELNETINC_04_08_2020-EX-1-JOINT FILING AGREEMENT,Exhibit 1\n\nJOINT FILING AGREEMENT\n\nThe und...
3,ADAMSGOLFINC_03_21_2005-EX-10.17-ENDORSEMENT A...,REDACTED COPY\n\nCONFIDENTIAL TREATMENT REQUES...
4,"KIROMICBIOPHARMA,INC_05_11_2020-EX-10.23-CONSU...",Exhibit 10.23 Corporate Address Fannin South P...


In [43]:
for i in [0, 100]:
    print(f"Contract {i} : {unique_contracts[i]['title']}")
    print(unique_contracts[i]['context'][:800])
    print("\n\n")

Contract 0 : LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGREEMENT
EXHIBIT 10.6

                              DISTRIBUTOR AGREEMENT

         THIS  DISTRIBUTOR  AGREEMENT (the  "Agreement")  is made by and between Electric City Corp.,  a Delaware  corporation  ("Company")  and Electric City of Illinois LLC ("Distributor") this 7th day of September, 1999.

                                    RECITALS

         A. The  Company's  Business.  The Company is  presently  engaged in the business  of selling an energy  efficiency  device,  which is  referred to as an "Energy  Saver"  which may be improved  or  otherwise  changed  from its present composition (the "Products").  The Company may engage in the business of selling other  products  or  other  devices  other  than  the  Products,  which  will be considered  Products if Distributor  exercises its option



Contract 100 : RUBIOSRESTAURANTSINC_03_31_2008-EX-10.75-SPONSORSHIP AGREEMENT
2008 Sponsorship Agreement- Renewal Sponsor

Exhibit 

In [44]:
eval_examples = [{
    "title" : row["title"],
    "question": row["question"],
    "answers": row["answers"]["text"],
}
for row in full_data]

print(len(eval_examples))
print(eval_examples[0])

26632
{'title': 'LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGREEMENT', 'question': 'Highlight the parts (if any) of this contract related to "Document Name" that should be reviewed by a lawyer. Details: The name of the contract', 'answers': ['DISTRIBUTOR AGREEMENT']}


In [45]:
has_answer = sum(1 for ex in eval_examples if len(ex["answers"]) > 0)
no_answer = len(eval_examples) - has_answer

print(f"with answer: {has_answer} ({has_answer/len(eval_examples) * 100:.1f}%)")
print(f"without answer: {no_answer} ({no_answer/len(eval_examples) * 100:.1f}%)")

with answer: 12424 (46.7%)
without answer: 14208 (53.3%)


In [46]:
categories = set()

for ex in eval_examples:
    q = ex["question"]

    if '"' in q:
        category = q.split('"')[1]
        categories.add(category)

print(f"Number of categories: {len(categories)}")
for c in sorted(categories):
    print(f"- {c}")

Number of categories: 41
- Affiliate License-Licensee
- Affiliate License-Licensor
- Agreement Date
- Anti-Assignment
- Audit Rights
- Cap On Liability
- Change Of Control
- Competitive Restriction Exception
- Covenant Not To Sue
- Document Name
- Effective Date
- Exclusivity
- Expiration Date
- Governing Law
- Insurance
- Ip Ownership Assignment
- Irrevocable Or Perpetual License
- Joint Ip Ownership
- License Grant
- Liquidated Damages
- Minimum Commitment
- Most Favored Nation
- No-Solicit Of Customers
- No-Solicit Of Employees
- Non-Compete
- Non-Disparagement
- Non-Transferable License
- Notice Period To Terminate Renewal
- Parties
- Post-Termination Services
- Price Restrictions
- Renewal Term
- Revenue/Profit Sharing
- Rofr/Rofo/Rofn
- Source Code Escrow
- Termination For Convenience
- Third Party Beneficiary
- Uncapped Liability
- Unlimited/All-You-Can-Eat-License
- Volume Restriction
- Warranty Duration


In [47]:
lengths = df["context"].str.len()
print("smaller contract length:")
print(df.loc[lengths.nsmallest(3).index, ["title"]])
print(f"length: {lengths.nsmallest(3).values}")

print("largest contract length:")
print(df.loc[lengths.nlargest(3).index, ["title"]])
print(f"length: {lengths.nlargest(3).values}")

smaller contract length:
                                                 title
442  RMRGROUPINC_01_22_2020-EX-99.1-JOINT FILING AG...
495  PcquoteComInc_19990721_S-1A_EX-10.11_6377149_E...
501  PRECIGEN,INC_01_22_2020-EX-99.1-JOINT FILING A...
length: [645 857 972]
largest contract length:
                                                 title
358  MANUFACTURERSSERVICESLTD_06_05_2000-EX-10.14-O...
157  CERES,INC_01_25_2012-EX-10.20-Collaboration Ag...
499  GOOSEHEADINSURANCE,INC_04_02_2018-EX-10.6-Fran...
length: [338211 335282 300768]


In [48]:
import re

sample_text = unique_contracts[0]["context"]

multi_spaces = re.findall(r" {3,}", sample_text)
multi_newlines = re.findall(r"\n{2,}", sample_text)

print(f"Number of multi-spaces (>=3): {len(multi_spaces)}")
print(f"Number of multi-newlines (>=2): {len(multi_newlines)}")
print(f"Text length: {len(sample_text)}")

Number of multi-spaces (>=3): 900
Number of multi-newlines (>=2): 121
Text length: 54290


In [49]:
# On utilise le tokenizer du modèle d'embedding qu'on prévoit d'utiliser (bge-small)
# pour avoir des longueurs réelles en tokens, pas une approximation par caractères
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("BAAI/bge-small-en-v1.5")

# Test sur les 3 contrats les plus courts et les 3 plus longs
sample_titles = list(lengths.nsmallest(3).index) + list(lengths.nlargest(3).index)

for idx in sample_titles:
    text = df.loc[idx, "context"]
    n_tokens = len(tokenizer.encode(text))
    print(f"{df.loc[idx, 'title'][:50]:<50} | {len(text):>7} chars | {n_tokens:>7} tokens")

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (71268 > 512). Running this sequence through the model will result in indexing errors


RMRGROUPINC_01_22_2020-EX-99.1-JOINT FILING AGREEM |     645 chars |     174 tokens
PcquoteComInc_19990721_S-1A_EX-10.11_6377149_EX-10 |     857 chars |     313 tokens
PRECIGEN,INC_01_22_2020-EX-99.1-JOINT FILING AGREE |     972 chars |     263 tokens
MANUFACTURERSSERVICESLTD_06_05_2000-EX-10.14-OUTSO |  338211 chars |   71268 tokens
CERES,INC_01_25_2012-EX-10.20-Collaboration Agreem |  335282 chars |   78593 tokens
GOOSEHEADINSURANCE,INC_04_02_2018-EX-10.6-Franchis |  300768 chars |   61676 tokens


In [50]:
import re

def clean_contract_text(text: str) -> str:

    text = re.sub(r"[ \t]{2,}", " ", text)

    text = re.sub(r"\n{3,}", "\n\n", text)

    text = "\n".join(line.strip() for line in text.split('\n'))

    return text.strip()

sample = unique_contracts[0]["context"]
cleaned = clean_contract_text(sample)

print(f"before cleaning: {len(sample)} chars")
print(f"after cleaning: {len(cleaned)} chars")
print(f"original text:\n{sample[:1000]}\n\n")
print(f"cleaned text:\n{cleaned[:1000]}")

before cleaning: 54290 chars
after cleaning: 38563 chars
original text:
EXHIBIT 10.6

                              DISTRIBUTOR AGREEMENT

         THIS  DISTRIBUTOR  AGREEMENT (the  "Agreement")  is made by and between Electric City Corp.,  a Delaware  corporation  ("Company")  and Electric City of Illinois LLC ("Distributor") this 7th day of September, 1999.

                                    RECITALS

         A. The  Company's  Business.  The Company is  presently  engaged in the business  of selling an energy  efficiency  device,  which is  referred to as an "Energy  Saver"  which may be improved  or  otherwise  changed  from its present composition (the "Products").  The Company may engage in the business of selling other  products  or  other  devices  other  than  the  Products,  which  will be considered  Products if Distributor  exercises its options pursuant to Section 7 hereof.

         B. Representations.  As an inducement to the Company to enter into this Agreement,  th

In [51]:
for c in unique_contracts:
    c["clean_context"] = clean_contract_text(c["context"])

reduction = [(len(c["context"]) - len(c["clean_context"])) / len(c["context"]) * 100 for c in unique_contracts]
print(f"average reduction in length after cleaning: {sum(reduction)/len(reduction):.1f}%")

average reduction in length after cleaning: 3.6%


In [52]:
df_cleaned = pd.DataFrame(unique_contracts)

print(df_cleaned["clean_context"].str.len().describe())

count       510.000000
mean      50592.280392
std       53909.954570
min         645.000000
25%       16134.500000
50%       32097.500000
75%       65329.750000
max      300304.000000
Name: clean_context, dtype: float64
